# [실습] LangGraph 워크플로우 패턴

LangGraph의 Graph 패턴을 활용하여 다양한 워크플로우를 구현합니다.

| 패턴 | 설명 | 사용 API |
|------|------|----------|
| Router | 입력을 분류하여 조건부 라우팅 | `add_conditional_edges()` |
| Fan-in / Fan-out | 여러 노드를 병렬 실행 후 결과 합산 | 다중 `add_edge(START, ...)` |
| Map-Reduce | 동적 개수의 워커를 병렬 생성 | `Send()` API |
| Evaluator-Optimizer | 생성하고 평가한 뒤 다듬는 반복 루프 | 조건부 엣지 + 루프 |

## 학습 목표

1. LangGraph `StateGraph`의 기본 구조(상태, 노드, 엣지) 복습
2. Router 패턴: `add_conditional_edges()`를 활용한 조건부 라우팅 구현
3. Map-Reduce 패턴: `Send()` API를 통한 동적 병렬 처리 구현
4. 각 패턴의 적절한 사용 상황 이해

## 0. 환경 설정

In [ ]:
import os
import operator
from typing import List
from typing_extensions import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv('.env', override=True)

from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langfuse.langchain import CallbackHandler
callbacks = [CallbackHandler()] if os.environ.get('LANGFUSE_PUBLIC_KEY') else []

# LLM 설정: 워크플로우는 짧고 반복적인 호출이 많아, 샘플링 폭을 좁힌 가벼운 vLLM 설정을 사용합니다.
from select_model import load_model

light_model = load_model(platform='vllm', enable_thinking=False, max_tokens=16384, temperature=0.7, top_p=0.80, presence_penalty=1.5, extra_body={"top_k": 20, "min_p": 0.0, "repetition_penalty": 1.0})
llm = light_model

print('환경 설정 완료')

---
## 1. Router 패턴

라우터는 State의 값을 기준으로 조건부 분기하여, 입력의 종류에 따라 다른 노드를 실행하는 패턴입니다.
주로 의도 분류(Intent Classification)에 활용됩니다.

노드의 리턴값은 역할에 따라 두 가지로 나뉩니다.
분기를 결정하는 분류 노드는 `response_format`으로 구조화된 출력(`structured_response`)을 받아 분류 결과를 State에 넣고,
답변처럼 텍스트가 산출물인 노드는 마지막 메시지의 `.text`를 State에 넣습니다.

### 시나리오: 보험 고객 문의 라우팅

고객 문의를 자동으로 분류하여, 분류 결과에 따라 전문 상담 채널로 연결합니다.

- PRODUCT: 보험 상품 안내는 상품 안내 채널로
- CLAIM: 보험금 청구/심사는 청구 안내 채널로
- CONTRACT: 계약 변경/해지는 계약 관리 채널로
- GENERAL: 일반 문의는 일반 상담 채널로

`route` 노드를 통해 문의를 분류하고, 분류 결과에 따라 다른 채널로 라우팅하는 패턴입니다.

In [ ]:
# 분류 결과 스키마
class Classification(BaseModel):
    category: Literal["PRODUCT", "CLAIM", "CONTRACT", "GENERAL"] = Field(
        description="고객 문의 분류 결과"
    )

# 라우터 상태 정의
class RouterState(TypedDict):
    query: str
    classification: str
    answer: str

print(f'RouterState 필드: {list(RouterState.__annotations__.keys())}')

In [ ]:
classifier = create_agent(
    llm,
    tools=[],
    system_prompt='''당신은 보험회사의 고객 문의를 분류하는 봇입니다.

1) 보험 상품 관련 질문 (가입, 상품 추천, 보장 내용 등): 'PRODUCT'
2) 보험금 청구/심사 관련 질문 (청구 방법, 필요 서류, 심사 기간 등): 'CLAIM'
3) 계약 변경 관련 질문 (계약 변경, 해지, 부활, 수익자 변경 등): 'CONTRACT'
4) 그 외 일반 질문: 'GENERAL'
''',
    response_format=Classification,
)


def route(state: RouterState):
    """고객 문의를 분류하는 라우터. structured_response로 분류 결과를 받는다."""
    result = classifier.invoke({'messages': [HumanMessage(f"고객 질문: {state['query']}")]})
    return {'classification': result['structured_response'].category}


def make_handler(system_prompt):
    """system_prompt만 다른 답변 노드를 만든다. 에이전트는 state의 query만 본다."""
    agent = create_agent(llm, tools=[], system_prompt=system_prompt)

    def handler(state: RouterState):
        result = agent.invoke({'messages': [HumanMessage(state['query'])]})
        return {'answer': result['messages'][-1].text}

    return handler


handle_product = make_handler('''당신은 보험 상품 안내 전문가입니다.
고객의 상황에 맞는 보험 상품을 친절하게 안내하세요.
실제 상품이 아닌 일반적인 상품 가이드라인을 안내합니다.
답변은 최대 2문단, 10문장 이내로 작성하세요.''')

handle_claim = make_handler('''당신은 보험금 청구 안내 전문가입니다.
필요 서류와 절차를 친절하게 안내하세요.
답변은 최대 2문단, 10문장 이내로 작성하세요.''')

handle_contract = make_handler('''당신은 계약 관리 전문가입니다.
변경 절차와 주의사항을 안내하세요.
답변은 최대 2문단, 10문장 이내로 작성하세요.''')

handle_general = make_handler('''당신은 친절한 보험 상담사입니다.
정확하고 친절하게 답변하세요.
답변은 최대 2문단, 10문장 이내로 작성하세요.''')


def route_decision(state: RouterState):
    """분류 결과에 따라 적절한 핸들러로 라우팅"""
    return {
        'PRODUCT': 'handle_product',
        'CLAIM': 'handle_claim',
        'CONTRACT': 'handle_contract',
    }.get(state['classification'], 'handle_general')

print('노드 함수 정의 완료')

In [ ]:
builder = StateGraph(RouterState)

# 노드 추가
builder.add_node("route", route)
builder.add_node("handle_product", handle_product)
builder.add_node("handle_claim", handle_claim)
builder.add_node("handle_contract", handle_contract)
builder.add_node("handle_general", handle_general)

# 엣지 설정
builder.add_edge(START, "route")
builder.add_conditional_edges("route", route_decision, {
    "handle_product": "handle_product",
    "handle_claim": "handle_claim",
    "handle_contract": "handle_contract",
    "handle_general": "handle_general",
})
builder.add_edge("handle_product", END)
builder.add_edge("handle_claim", END)
builder.add_edge("handle_contract", END)
builder.add_edge("handle_general", END)

router_graph = builder.compile()

# 그래프 시각화
router_graph

In [ ]:
# 상품 문의 테스트
query = '30대 직장인인데, 암보험에 가입하려고 합니다. 어떤 상품이 좋을까요?'
result = router_graph.invoke({'query': query},
    config={'callbacks': callbacks})
print(f'분류: {result["classification"]}')
print(f'답변:\n{result["answer"]}')

In [ ]:
# 청구 문의 테스트
query = '교통사고로 입원했는데 보험금 청구하려면 어떻게 해야 하나요?'
result = router_graph.invoke(
    {'query': query},
    config={'callbacks': callbacks}
)
print(f'분류: {result["classification"]}')
print(f'답변:\n{result["answer"]}')

In [ ]:
# 일반 문의 테스트
query = '오늘 날씨가 어때요?'
result = router_graph.invoke(
    {'query': query},
    config={'callbacks': callbacks}
)
print(f'분류: {result["classification"]}')
print(f'답변:\n{result["answer"]}')

---
## 2. Fan-in / Fan-out 패턴

조건분기가 아닌 일반 그래프의 경우에도, 여러 노드가 동시에 실행될 수 있습니다.
Fan-in / Fan-out 패턴은 분기 없이도 노드를 병렬적으로 실행한 후, 결과를 합치는 패턴입니다.

### 시나리오: 종합 코드 리뷰

하나의 코드를 세 가지 관점에서 동시에 리뷰하고, 결과를 종합합니다.
세 리뷰 노드는 서로의 결과를 보지 못하고, State의 code만 봅니다.

- 성능 리뷰어: 시간 복잡도, 메모리 효율성, 알고리즘 적절성 분석
- 보안 리뷰어: SQL Injection, XSS, 인증 관련 취약점 분석
- 유지보수성 리뷰어: 네이밍, 코드 구조, 테스트 용이성 분석

In [ ]:
class CodeReviewState(TypedDict):
    code: str
    performance_review: str
    security_review: str
    maintainability_review: str
    final_review: str

print(f'CodeReviewState 필드: {list(CodeReviewState.__annotations__.keys())}')

In [ ]:
def make_reviewer(field, system_prompt):
    """관점별 리뷰 노드를 만든다. 에이전트는 state의 code만 본다."""
    agent = create_agent(llm, tools=[], system_prompt=system_prompt)

    def reviewer(state: CodeReviewState):
        result = agent.invoke({'messages': [HumanMessage(f"코드:\n```\n{state['code']}\n```")]})
        return {field: result['messages'][-1].text}

    return reviewer


review_performance = make_reviewer('performance_review', '''당신은 시니어 백엔드 개발자입니다.
주어진 코드를 성능 관점에서 리뷰하세요.

분석 항목:
- 시간 복잡도 / 공간 복잡도
- 불필요한 연산이나 반복
- 메모리 누수 가능성
- DB 쿼리 최적화 (해당 시)

간결하게 작성하세요.''')

review_security = make_reviewer('security_review', '''당신은 보안 전문가입니다.
주어진 코드를 보안 관점에서 리뷰하세요.

분석 항목:
- SQL Injection, XSS 등 보안 취약점
- 인증/인가(비밀번호, API 키 등) 처리
- 입력값 검증 여부

간결하게 작성하세요.''')

review_maintainability = make_reviewer('maintainability_review', '''당신은 클린 코드 전문가입니다.
주어진 코드를 유지보수성 관점에서 리뷰하세요.

분석 항목:
- 변수/함수 네이밍
- 코드 구조와 가독성
- 단일 책임 원칙 준수 여부
- 테스트 용이성

간결하게 작성하세요.''')


synthesizer = create_agent(
    llm,
    tools=[],
    system_prompt='''세 관점의 코드 리뷰를 종합하여 최종 리뷰를 작성하세요.
각 관점의 주요 지적사항을 정리하고, 보완 우선순위 3개를 제시하세요.''',
)


def summarize_review(state: CodeReviewState):
    """세 관점의 리뷰를 종합하는 노드. 원본 코드와 세 리뷰를 모두 본다."""
    report = f'''## 원본 코드
```
{state['code']}
```

## 성능 리뷰
{state['performance_review']}

## 보안 리뷰
{state['security_review']}

## 유지보수성 리뷰
{state['maintainability_review']}'''
    result = synthesizer.invoke({'messages': [HumanMessage(report)]})
    return {'final_review': result['messages'][-1].text}

print('코드 리뷰 노드 정의 완료')

In [ ]:
builder = StateGraph(CodeReviewState)

builder.add_node("review_performance", review_performance)
builder.add_node("review_security", review_security)
builder.add_node("review_maintainability", review_maintainability)
builder.add_node("summarize_review", summarize_review)

# Fan-out: START에서 세 리뷰어로 동시 분기
builder.add_edge(START, "review_performance")
builder.add_edge(START, "review_security")
builder.add_edge(START, "review_maintainability")

# Fan-in: 세 리뷰어 결과가 모두 모이면 summarize_review로 합류
builder.add_edge("review_performance", "summarize_review")
builder.add_edge("review_security", "summarize_review")
builder.add_edge("review_maintainability", "summarize_review")
builder.add_edge("summarize_review", END)

code_review_graph = builder.compile()

# 그래프 시각화
code_review_graph

In [ ]:
from stream_utils import stream_workflow

# 보안 취약점이 있는 코드 (보험료 계산 API 예시)
sample_code = '''
def calculate_premium(user_id, product_code):
    # DB에서 사용자 정보 조회
    query = f"SELECT * FROM users WHERE id = {user_id}"
    user = db.execute(query).fetchone()

    # 상품 정보 조회
    product = db.execute(f"SELECT * FROM products WHERE code = '{product_code}'").fetchone()

    # 나이 계산
    age = 2025 - int(user['birth_year'])

    # 보험료 계산
    base_premium = product['base_price']
    if age > 40:
        premium = base_premium * 1.5
    else:
        premium = base_premium

    print(f"User {user['name']} premium: {premium}")
    return premium
'''

# 세 리뷰어가 병렬 실행되며 ▶ 노드 시작 마커가 인터리브되어 출력됩니다.
# Fan-in 노드(summarize_review)는 셋이 모두 끝나야 시작됩니다.
result = await stream_workflow(
    code_review_graph,
    {'code': sample_code},
    config={'callbacks': callbacks},
)

In [ ]:
from IPython.display import Markdown

# 최종 종합 리뷰 확인
Markdown(result['final_review'])

---
## 3. Map-Reduce 패턴

Fan-in/Fan-out에서는 정해진 개수만큼의 노드를 병렬 실행했는데요.
만약 실행 과정에서 병렬 처리 노드의 개수가 정해진다면 어떻게 해야 할까요?

Map-Reduce는 이런 과정에 활용되는 패턴입니다.

기술 비교 문서와 같이, 여러 챕터로 구성된 보고서를 작성하는 로직을 구현해 보겠습니다.

1. 먼저, Orchestrator가 주어진 주제를 분석하여 문서의 섹션 구조(`DocumentPlan`)를 기획합니다.
2. 이후, 각 섹션에 대한 글을 작성하는 Workers가 섹션 개수만큼 실행됩니다.
   이는 Orchestrator에서 `Send()`로 write_section 노드를 병렬 호출하는 방식입니다.
3. 전체 섹션의 리스트를 Synthesizer에서 종합하여 최종 문서를 완성합니다.

```
User -> plan_document -> [Send x N] -> write_section (xN 병렬) -> synthesize_document -> END
```

In [ ]:
# 섹션 스키마
class Section(BaseModel):
    title: str = Field(description="섹션 제목")
    description: str = Field(description="이 섹션에서 다룰 주요 내용")

class DocumentPlan(BaseModel):
    sections: List[Section] = Field(description="문서의 섹션 목록")

# 메인 그래프 상태
class MapReduceState(TypedDict):
    topic: str                                          # 문서 주제
    sections: list[Section]                             # 기획된 섹션 목록
    completed_sections: Annotated[list, operator.add]   # 워커 결과를 합산하는 리듀서
    final_document: str                                 # 최종 문서

# 워커 상태 (각 워커는 독립적인 상태를 가짐)
class WorkerState(TypedDict):
    topic: str
    section: Section
    completed_sections: Annotated[list, operator.add]

print('Map-Reduce 스키마 정의 완료')
print(f'  MapReduceState 필드: {list(MapReduceState.__annotations__.keys())}')
print(f'  WorkerState 필드: {list(WorkerState.__annotations__.keys())}')

In [ ]:
planner = create_agent(
    llm,
    tools=[],
    system_prompt='''당신은 기술 문서 기획 전문가입니다.
주어진 주제에 대해 문서의 섹션 구조를 기획하세요.

규칙:
- 섹션은 3~5개로 제한
- 각 섹션의 제목과 다룰 내용을 명확히 기술''',
    response_format=DocumentPlan,
)

section_writer = create_agent(
    llm,
    tools=[],
    system_prompt='''당신은 기술 문서 작성 전문가입니다.
주어진 섹션을 마크다운 형식으로 작성하세요.
- 소제목은 ## 레벨, 하위 항목은 ### 레벨로 표기
- 근거와 데이터를 포함하여 3~5문단으로 작성''',
)


def plan_document(state: MapReduceState):
    """Orchestrator: 문서 구조를 기획하는 노드. structured_response로 섹션 목록을 받는다."""
    result = planner.invoke({'messages': [HumanMessage(f"주제: {state['topic']}")]})
    return {'sections': result['structured_response'].sections}


# Send() API: LangGraph에서, 각 노드를 병렬 호출하는 API
def assign_section_workers(state: MapReduceState):
    """Send() API로 각 섹션을 별도 워커에게 분배"""
    return [
        Send("write_section", {"section": s, "topic": state["topic"]})
        for s in state["sections"]
    ]


def write_section(state: WorkerState):
    """Worker: 섹션 하나를 작성하는 노드. 전체 문서가 아니라 자기 섹션과 주제만 본다."""
    section = state['section']
    result = section_writer.invoke({'messages': [HumanMessage(
        f'전체 주제: {state["topic"]}\n\n섹션 제목: {section.title}\n섹션 설명: {section.description}')]})
    return {"completed_sections": [result['messages'][-1].text]}


def synthesize_document(state: MapReduceState):
    """Synthesizer: 모든 섹션을 종합"""
    completed = "\n\n---\n\n".join(state["completed_sections"])
    return {"final_document": f"# {state['topic']}\n\n{completed}"}

print('Map-Reduce 노드 정의 완료')

In [ ]:
builder = StateGraph(MapReduceState)

builder.add_node(plan_document)
builder.add_node(write_section)
builder.add_node(synthesize_document)

builder.add_edge(START, "plan_document")
# Send()로 동적 워커 생성: 섹션 수만큼 write_section 인스턴스가 병렬 실행됨
builder.add_conditional_edges("plan_document", assign_section_workers, ["write_section"])
builder.add_edge("write_section", "synthesize_document")
builder.add_edge("synthesize_document", END)

mapreduce_graph = builder.compile()

# 그래프 시각화
mapreduce_graph

In [ ]:
# stream 모드로 각 노드의 실행 과정 확인
for data in mapreduce_graph.stream(
    {"topic": "REST API vs GraphQL 기술 비교"},
    config={'callbacks': callbacks},
    stream_mode='updates',
):
    for node_name, output in data.items():
        if node_name == "plan_document":
            sections = output.get("sections", [])
            print(f'\n[plan_document] {len(sections)}개 섹션 기획됨')
            for s in sections:
                print(f'  - {s.title}: {s.description[:50]}...')
        elif node_name == "write_section":
            cs = output.get("completed_sections", [])
            for c in cs:
                title = c.split('\n')[0]
                print(f'[write_section] {title} ({len(c)}자)')
        elif node_name == "synthesize_document":
            doc = output.get("final_document", "")
            print(f'\n[synthesize_document] 최종 문서 생성 ({len(doc)}자)')

In [ ]:
from IPython.display import Markdown

# 최종 문서 확인 (마지막 data에 결과 저장)
final_doc = data.get('synthesize_document', {}).get('final_document', '')

with open('report.md', 'w', encoding='utf-8') as f:
    f.write(final_doc)
print("최종 문서가 'report.md'로 저장되었습니다.")


## [실습] 서론/결론을 포함한 Map-Reduce 패턴

현재 Map-Reduce 패턴으로 보고서를 작성하면,
서론과 결론의 일관성이 중간 부분과 일관성이 떨어지는 문제가 생길 수 있습니다.

Synthesize Documents 노드가 서론/결론을 작성해 최종 결과물을 만들도록 그래프를 구성해 보세요.

중간 섹션을 병렬로 생성한 다음 서론을 생성하고, 마지막으로 결론을 생성합니다.

---
## 4. Evaluator-Optimizer 패턴

LLM 결과를 평가 기준으로 점검하고 필요할 때 다시 생성하면 기준에 더 잘 맞는 결과를 얻을 수 있습니다.

### 시나리오: 챗봇 시스템 프롬프트 최적화

챗봇의 시스템 프롬프트를 생성하고, 평가 기준에 맞게 다듬습니다.

몇 가지 평가 기준에 맞게 자동으로 다듬습니다.

In [ ]:
class PromptFeedback(BaseModel):
    grade: Literal["approved", "needs_improvement"] = Field(
        description="프롬프트가 평가 기준을 충족하는지 판단"
    )
    feedback: str = Field(
        description="보완이 필요한 경우, 구체적인 조정 방향을 제시"
    )

class PromptOptimizerState(TypedDict):
    instruction: str       # 어떤 챗봇의 프롬프트를 만들 것인지
    criteria: str          # 좋은 프롬프트를 위한 평가 가이드라인
    prompt: str            # 생성된 시스템 프롬프트
    feedback: str          # 평가자의 피드백
    approved: str          # 승인 여부
    num_revision: int      # 평가 반복 횟수

print('Evaluator-Optimizer 스키마 정의 완료')

In [ ]:
guideline_writer = create_agent(
    llm,
    tools=[],
    system_prompt='주어진 목적에 맞는 시스템 프롬프트를 작성하려고 합니다. 평가 기준에 맞는 프롬프트를 만들기 위한 가이드라인을 개조식으로, 8문장으로 출력하세요.',
)

prompt_writer = create_agent(
    llm,
    tools=[],
    system_prompt='당신은 전문 챗봇 시스템 프롬프트 설계자입니다. 주어진 용도에 맞는 챗봇 시스템 프롬프트를 작성하세요. 프롬프트만 출력하세요.',
)

evaluator = create_agent(
    llm,
    tools=[],
    system_prompt='챗봇 시스템 프롬프트를 평가 기준에 따라 평가합니다.',
    response_format=PromptFeedback,
)


def prompt_guideline(state: PromptOptimizerState):
    """목적에 맞는 평가 가이드라인을 생성하는 노드. instruction만 본다."""
    result = guideline_writer.invoke({'messages': [HumanMessage(f"목적: {state['instruction']}")]})
    return {"criteria": result['messages'][-1].text}


def prompt_generator(state: PromptOptimizerState):
    """챗봇 시스템 프롬프트 생성 노드. 피드백이 있으면 현재 프롬프트와 피드백을 함께 본다."""
    instruction = state['instruction']
    feedback = state.get('feedback', '')
    current_prompt = state.get('prompt', '')

    if feedback:
        # 피드백이 있으면 현재 프롬프트를 다듬음
        gen_instruction = f'''현재 챗봇 시스템 프롬프트와 피드백입니다.
피드백에 맞게 프롬프트를 다듬으세요.

---
현재 프롬프트:
{current_prompt}

피드백:
{feedback}
---

챗봇 용도: {instruction}'''
    else:
        gen_instruction = instruction

    result = prompt_writer.invoke({'messages': [HumanMessage(gen_instruction)]})
    return {"prompt": result['messages'][-1].text}


def prompt_evaluator(state: PromptOptimizerState):
    """프롬프트를 평가 기준으로 판정하는 노드. structured_response로 판정을 받는다."""
    num_revision = state.get('num_revision', 0) + 1

    result = evaluator.invoke({'messages': [HumanMessage(f'''다음 챗봇 시스템 프롬프트를 평가 기준으로 판정하세요.

---
챗봇 용도: {state['instruction']}
---
시스템 프롬프트:
{state['prompt']}
---

평가 기준:
{state['criteria']}

모든 기준을 충족하면 'approved', 아니면 'needs_improvement'로 판정하세요.''')]})
    decision = result['structured_response']
    return {
        "approved": decision.grade,
        "feedback": decision.feedback,
        "num_revision": num_revision,
    }


def route_prompt(state: PromptOptimizerState):
    """평가 결과에 따라 루프 제어 (최대 2회 반복)"""
    if state["approved"] == "approved" or state.get('num_revision', 0) > 2:
        return "Approved"
    return "Needs Improvement"

print('Evaluator-Optimizer 노드 정의 완료')

In [ ]:
builder = StateGraph(PromptOptimizerState)

builder.add_node(prompt_guideline)
builder.add_node(prompt_generator)
builder.add_node(prompt_evaluator)

builder.add_edge(START, "prompt_guideline")
builder.add_edge("prompt_guideline", "prompt_generator")

builder.add_edge("prompt_generator", "prompt_evaluator")
builder.add_conditional_edges(
    "prompt_evaluator",
    route_prompt,
    {"Approved": END, "Needs Improvement": "prompt_generator"}
)

evalopt_graph = builder.compile()

# 그래프 시각화
evalopt_graph

In [ ]:
# 보험 상품 추천 챗봇의 시스템 프롬프트 최적화
for data in evalopt_graph.stream(
    {'instruction': '보험 상품 추천 챗봇'},
    config={'callbacks': callbacks},
    stream_mode='updates',
):
    if 'prompt_guideline' in data:
        print("=== 프롬프트 가이드라인 ===")
        print(data['prompt_guideline']['criteria'])
    elif 'prompt_generator' in data:
        print("=== 생성된 프롬프트 ===")
        print(data['prompt_generator']['prompt'])
    else:
        ev = data['prompt_evaluator']
        print(f"\n=== 평가 결과: {ev['approved']} ({ev['num_revision']}회차) ===")
        print(ev['feedback'])
    print('-' * 50)

In [ ]:
# 취업 준비생 상담 챗봇의 시스템 프롬프트 최적화
for data in evalopt_graph.stream(
    {'instruction': '취업 준비생 대상의 상담 챗봇'},
    config={'callbacks': callbacks},
    stream_mode='updates',
):
    if 'prompt_guideline' in data:
        print("=== 프롬프트 가이드라인 ===")
        print(data['prompt_guideline']['criteria'])
    elif 'prompt_generator' in data:
        print("=== 생성된 프롬프트 ===")
        print(data['prompt_generator']['prompt'])
    else:
        ev = data['prompt_evaluator']
        print(f"\n=== 평가 결과: {ev['approved']} ({ev['num_revision']}회차) ===")
        print(ev['feedback'])
    print('-' * 50)

---
## 5. 패턴 비교 및 선택 가이드

### 4가지 패턴 비교

| 관점 | Router | Fan-in/Fan-out | Map-Reduce | Evaluator-Optimizer |
|------|--------|----------------|------------|---------------------|
| 구조 | 조건 분기 | 정적 병렬 후 합류 | 동적 병렬 후 합산 | 반복 루프 |
| 사용 API | `add_conditional_edges()` | 다중 `add_edge()` | `Send()` | 조건부 엣지 + 루프 |
| 워커 수 | 1개 (선택) | 고정 (빌드 시점) | 동적 (런타임) | 고정 (2개) |
| 적합 사례 | 의도 분류, A/B 라우팅 | 다관점 분석, 앙상블 | 문서 생성, 데이터 파이프라인 | 평가 기반 프롬프트 다듬기 |

### 언제 어떤 패턴을 사용할까?

- Router: 입력의 종류에 따라 다른 처리가 필요할 때 (챗봇 의도 분류, 문의 유형별 처리)
- Fan-in / Fan-out: 하나의 입력을 여러 고정된 관점에서 처리할 때 (코드 리뷰, 다면 평가)
- Map-Reduce: 작업을 동적 개수의 하위 작업으로 분할할 때 (문서 생성, 대량 데이터 처리)
- Evaluator-Optimizer: 출력을 평가 기준에 맞게 반복해서 다듬어야 할 때 (프롬프트 조정, 보고서 다듬기)

### 단일 에이전트 vs 워크플로우

| 상황 | 추천 방식 |
|------|-----------|
| 도구 수가 적고, 유연한 도구 선택이 필요 | 단일 에이전트 (`create_agent()`) |
| 처리 흐름이 명확하고, 각 단계가 독립적 | 워크플로우 (`StateGraph`) |
| 평가 기준 통과가 필요하고, 반복 다듬기가 필요 | Evaluator-Optimizer 패턴 |
| 대량의 하위 작업을 병렬 처리해야 함 | Map-Reduce 패턴 |

---
## 정리

이번 실습에서 학습한 내용:

1. Router 패턴: `add_conditional_edges()`로 입력을 분류하여 조건별 라우팅
2. Fan-in / Fan-out 패턴: 여러 노드를 동시 실행하고 결과를 합류
3. Map-Reduce 패턴: `Send()` API로 동적인 워커를 생성하여 병렬 처리
4. Evaluator-Optimizer 패턴: 평가와 반복 다듬기로 출력이 기준을 통과하도록 구성
5. 리턴값 규칙: 분기/기획/평가 노드는 `structured_response`, 답변/작성 노드는 마지막 메시지의 `.text`를 State에 저장
6. 패턴 선택 기준: 태스크 특성에 따라 적절한 워크플로우 패턴 판단